# IMPORT LIBRARY

In [ ]:
import os
import random
import datetime
import json
import pandas   as pd
import numpy    as np

from tqdm                               import tqdm
from preProcessingDataTEP               import PreProcessData
from utility.matpilotDraw               import MatPlot

with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["tep"]

# LOADING DATA

In [ ]:
# This function is used to load and save data into H5 file
# preData     = PreProcessData(config=myConfig)
# outputDir   = os.path.expanduser(preData.config["dataDir"])    
# oriData     = preData.loadCSVDataFile(dir               = outputDir, 
#                                       removeHigherThr   = True, 
#                                       columnNames       = ["Stripper Level","Stripper Level","Stripper Level"],
#                                       thr               = ["down", "down", "up"]) 




# preData.save2H5Format(dir = outputDir, data = oriData)

In [ ]:
plot               = MatPlot() 

# ----------------------    
# Reactor, Separator,Stripper
# ----------------------
np.random.seed(1)
random.seed(1)

preData     = PreProcessData(config=myConfig)
outputDir   = os.path.expanduser(preData.config["dataDir"])
rawData             = preData.loadDataFromH5Format(dir = outputDir, name="data/newTeps.h5", case= "all", reduceSampleRate = False)

# withRULData         = preData.addRULAndEnvironment(rawData)
# cleanData           = preData.cleanData(   rawSignal           =   withRULData,
#                                                 columnName          =   preData.allWithoutLabel,
#                                                 getMovingAverage    =   False,
#                                                 flipToDown          =   True)
# # ---------------------------
# # For label, flip only
# cleanData           = preData.cleanData(rawSignal              =   cleanData,
#                                         columnName          =   preData.degraLiq,
#                                         getMovingAverage    =   False,
#                                         flipToDown          =   True)
# print(rawData[1])


In [ ]:
# Draw the data
curTrajectory = preData.getTrajectoryAt(inData= rawData, case = 3, idxTrajectory= 2)
for feature in preData.allHeadline:
    plot.draw2D(curTrajectory["Time"],curTrajectory[feature], title= feature)

In [ ]:
for idx in range(1):
    curTrajectory = preData.getTrajectoryAt(inData= rawData, case = 2, idxTrajectory= idx + 1)
    for idxHeadline in ['Reactor Level', 'Product Sep Level', 'Stripper Level']: 
        plot.draw2D(curTrajectory["Time"],curTrajectory[idxHeadline], title=idxHeadline)


In [ ]:
cleanData           = preData.cleanData(rawSignal               =   withoutInitRawData,
                                            columnName          =   preData.allWithoutLabel,
                                            getMovingAverage    =   True,
                                            flipToDown          =   True)
# ---------------------------
# For label, flip only
cleanData           = preData.cleanData(rawSignal           =   cleanData,
                                        columnName          =   preData.degraLiq,
                                        getMovingAverage    =   False,
                                        flipToDown          =   True)

movedData           = preData.movingToPeak(cleanData)


In [ ]:
# Draw the data
curTrajectory = preData.getTrajectoryAt(inData= cleanData, case = 2, idxTrajectory= 1)
for feature in preData.allHeadline:
    plot.draw2D(curTrajectory["Time"],curTrajectory[feature],title=feature)

In [ ]:
# Draw the data
print(cleanData)

### d. Normalize the data

In [ ]:
norData             = preData.normalizeDataAllModes(movedData)                # Normalize the data
# print(norData)

In [ ]:
# Draw the data
curTrajectory = preData.getTrajectoryAt(inData = norData, case = 2, idxTrajectory = 1)
for feature in preData.allHeadline:
    print(feature)
    plot.draw2D(curTrajectory["Time"],curTrajectory[feature],title=feature)

In [ ]:
plot = MatPlot()

plot.subPlotMultiImg(inY = hi["reactor"],  numRow=20, numCol=5)

### g. Split into each sequence

In [ ]:

seqTrainXY   = preData.getTensorData(norData, component =   "reactor", case = 1 )
# seqValXY   = preData.splitAllDataIntoSequences(valXY)  

# print(seqTrainXY[0].shape,seqTrainXY[1].shape)
# print(seqTrainXY[0][2011][:,0])

### h. Visualize the tensor data

In [ ]:

# Creating an array from 1 to 3000
#time = np.arange(1, 10)
# t= 0:200
# plot.draw1D(yCoor=seqTrainXY[1][0:t])
plot.draw1D(yCoor=seqTrainXY[0][2011][:,3])
            


# 4. Model and training

### To tensor

In [ ]:
tensorTrainXY       = data.TensorDataset(seqTrainXY[0],seqTrainXY[1])
tensorValXY         = data.TensorDataset(seqValXY[0],seqValXY[1])

In [ ]:
train_loader = data.DataLoader(tensorTrainXY,
                                batch_size  =   config.batch_size,
                                shuffle     =   False, 
                                num_workers =   config.num_workers)
        
val_loader = data.DataLoader(  tensorValXY, 
                                batch_size  =   config.val_batch_size, 
                                shuffle     =   False, 
                                num_workers =   config.num_workers)